# Logistic Regression Training (PySpark ML)

Train logistic regression models for network intrusion detection using PySpark ML to handle large datasets (10GB+).

## Strategies:
1. **Balanced Data**: Train on oversampled balanced dataset
2. **Class Weighting**: Use class weights to handle imbalance

In [ ]:
import importlib
import os
import sys
sys.path.append('..')

from pathlib import Path

# Reload to pick up training_utils changes without kernel restart
import notebooks.training_utils
importlib.reload(notebooks.training_utils)
from notebooks.training_utils import (
    load_training_data_pyspark,
    train_and_evaluate_pyspark,
    save_models_pyspark,
    print_summary_pyspark
)

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

# Stop any existing Spark session
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("Stopped existing Spark session")
except:
    pass

# Create Spark session optimised for M4 Mac (16 GB RAM, 10-core)
# driver.memory=6g (not 8g): leaves ~4-5 GB for macOS + Python on a 16 GB machine,
# reducing OS memory pressure and the risk of swapping during training.
# MEMORY_AND_DISK caching is safe — 176 GB disk available for spill.
spark = SparkSession.builder \
    .appName("LogisticRegressionTraining") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.default.parallelism", "20") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .getOrCreate()

print(f"✓ Spark initialized")
print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 20:50:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ Spark initialized
Spark Version: 3.5.6
Spark UI: http://mac.home:4040


## 1. Load Data

In [2]:
# Load data (verbose=False skips row counts for faster loading; restart kernel if TypeError)
train_orig_vec, train_balanced_vec, test_vec, feature_cols, project_root = load_training_data_pyspark(spark, verbose=False)


Loading training and test data from Parquet with PySpark...
✓ Data loaded (333 features, counts deferred for speed)
✓ Assembled feature vectors


26/02/28 20:50:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
# Data is already prepared by the utility function
print(f"✓ Ready for training with {len(feature_cols)} features")

✓ Ready for training with 333 features


## 2. Train Models

In [4]:
### Strategy 1: Balanced Data

In [ ]:
# Strategy 1: Balanced Data
model_params = {
    'featuresCol': 'features',
    'labelCol': 'label',
    'maxIter': 50,           # Reduced from 100 — LR typically converges in 20-40 iters
    'tol': 1e-3,             # Loosened from 1e-4 — stops earlier with negligible accuracy loss
    'regParam': 0.01,
    'elasticNetParam': 0.0,  # L2 regularization
    'family': 'binomial',
    'aggregationDepth': 4,   # Speeds up treeAggregate for many features/partitions
}

model_balanced, metrics_balanced, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_balanced_vec,
    test_vec,
    "Logistic Regression - Balanced Data Strategy",
    use_class_weights=False
)


26/02/28 20:50:56 WARN CacheManager: Asked to cache already cached data.


TRAINING: Logistic Regression - Balanced Data Strategy


26/02/28 20:51:12 WARN MemoryStore: Not enough space to cache rdd_23_2 in memory! (computed 451.6 MiB so far)
26/02/28 20:51:12 WARN BlockManager: Persisting block rdd_23_2 to disk instead.
26/02/28 20:51:12 WARN MemoryStore: Not enough space to cache rdd_23_1 in memory! (computed 451.6 MiB so far)
26/02/28 20:51:12 WARN BlockManager: Persisting block rdd_23_1 to disk instead.
26/02/28 20:51:12 WARN MemoryStore: Not enough space to cache rdd_23_3 in memory! (computed 131.0 MiB so far)
26/02/28 20:51:12 WARN BlockManager: Persisting block rdd_23_3 to disk instead.
26/02/28 20:51:13 WARN MemoryStore: Not enough space to cache rdd_23_7 in memory! (computed 259.1 MiB so far)
26/02/28 20:51:13 WARN BlockManager: Persisting block rdd_23_7 to disk instead.
26/02/28 20:51:15 WARN MemoryStore: Not enough space to cache rdd_23_5 in memory! (computed 707.4 MiB so far)
26/02/28 20:51:15 WARN BlockManager: Persisting block rdd_23_5 to disk instead.
26/02/28 20:51:15 WARN MemoryStore: Not enough spa

Py4JJavaError: An error occurred while calling o15118.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 8 in stage 7.0 failed 1 times, most recent failure: Lost task 8.0 in stage 7.0 (TID 22) (mac.home executor driver): com.esotericsoftware.kryo.KryoException: java.io.IOException: No space left on device
Serialization trace:
buffers (org.apache.spark.sql.execution.columnar.DefaultCachedBatch)
	at com.esotericsoftware.kryo.io.Output.flush(Output.java:188)
	at com.esotericsoftware.kryo.io.Output.require(Output.java:164)
	at com.esotericsoftware.kryo.io.Output.writeBytes(Output.java:251)
	at com.esotericsoftware.kryo.io.Output.writeBytes(Output.java:237)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ByteArraySerializer.write(DefaultArraySerializers.java:49)
	at com.esotericsoftware.kryo.serializers.DefaultArraySerializers$ByteArraySerializer.write(DefaultArraySerializers.java:38)
	at com.esotericsoftware.kryo.Kryo.writeObjectOrNull(Kryo.j

26/02/28 23:20:46 WARN BlockManager: Putting block rdd_23_6 failed due to exception com.esotericsoftware.kryo.KryoException: java.io.IOException: No space left on device
Serialization trace:
buffers (org.apache.spark.sql.execution.columnar.DefaultCachedBatch).
26/02/28 23:20:46 WARN BlockManager: Block rdd_23_6 could not be removed as it was not found on disk or in memory
26/02/28 23:20:46 WARN TaskSetManager: Lost task 6.0 in stage 7.0 (TID 20) (mac.home executor driver): TaskKilled (Stage cancelled: Job aborted due to stage failure: Task 8 in stage 7.0 failed 1 times, most recent failure: Lost task 8.0 in stage 7.0 (TID 22) (mac.home executor driver): com.esotericsoftware.kryo.KryoException: java.io.IOException: No space left on device
Serialization trace:
buffers (org.apache.spark.sql.execution.columnar.DefaultCachedBatch)
	at com.esotericsoftware.kryo.io.Output.flush(Output.java:188)
	at com.esotericsoftware.kryo.io.Output.require(Output.java:164)
	at com.esotericsoftware.kryo.io.O

In [ ]:
# Strategy 2: Class Weighting
model_weighted, metrics_weighted, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_orig_vec,
    test_vec,
    "Logistic Regression - Class Weight Strategy",
    use_class_weights=True
)

## 3. Save Models

In [ ]:
# Save models and metrics
save_models_pyspark(model_balanced, model_weighted, metrics_balanced, metrics_weighted, 'lr', project_root)

## 4. Summary

In [ ]:
# Print comparison summary
print_summary_pyspark(metrics_balanced, metrics_weighted, "Logistic Regression")

In [ ]:
# Stop Spark session when done
# spark.stop()